[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/drigaa/rEVA_CE4IB/HEAD?urlpath=%2Fdoc%2Ftree%2Fcasestudy.ipynb)

![thermal-network](images/Screenshot%202026-10-29%20093356.png)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from dm4bem import read_epw, sol_rad_tilt_surf

In [ ]:
filename = Path("weather_data/Hombrechtikon/CHE_ZH_Hombrechtikom.068510_TMYx.2011-2025.epw")
data, meta = read_epw(filename, coerce_year=None)

In [ ]:
air = {
    'Density': 1.2,
    'Specific heat': 1000
}

In [ ]:
l = 3

concrete = {
    'Conductivity': 1.400,
    'Density': 2300.0,
    'Specific heat': 880,
    'Width': 0.15,
    'Surface': l**2
}

insulation = {
    'Conductivity': 0.040,
    'Density': 16.0,
    'Specific heat': 1210,
    'Width': 0.25,
    'Surface': l**2
}

glass = {
    'U-Value': 0.6,
    'Surface': l**2
}

wall = pd.DataFrame.from_dict(
    {
        'Layer_out': insulation,
        'Layer_in': concrete,
        'window': glass
    },
    orient='index')

wall

In [ ]:
# convection coefficients, W/(m²·K)
h = pd.DataFrame([{'in': 4., 'out': 10.}], index=['h'])
h

In [ ]:
## P-controler gain
Kp = 0

In [ ]:
number_of_nodes = 10
number_of_conductances = 14
G_diag = np.zeros(number_of_conductances)

G_diag[[1, 2]] = (wall["Conductivity"] / (wall["Width"] / 2) * wall["Surface"]).Layer_out
G_diag[[3, 4, 7 ,8]] =  (wall["Conductivity"] / (wall["Width"] / 2) * wall["Surface"]).Layer_in

G_diag[[0]] = h['out'] * wall["Surface"].Layer_out
G_diag[[5, 6, 9]] = h['in'] * wall["Surface"].Layer_in

G_diag[[12, 13]] = (wall['U-Value'] * wall["Surface"]).window

G_diag[[10, 11]] = Kp

G = np.diag(G_diag)

In [ ]:
C_diag = np.zeros(number_of_nodes)

In [ ]:
wall['Width'] * wall['Surface'] * wall['Density'] * wall["Specific heat"]

In [ ]:
C_diag[1] = (wall['Width'] * wall['Surface'] * wall['Density'] * wall["Specific heat"]).Layer_out
C_diag[[3, 7]] = (wall['Width'] * wall['Surface'] * wall['Density'] * wall["Specific heat"]).Layer_in
C_diag[[5, 9]] = air["Density"] * l**3 * air["Specific heat"]
C = np.diag(C_diag)